# Huấn luyện mô hình Intent Recognition tiếng Việt trên Google Colab (v2)

Notebook này huấn luyện mô hình phân loại ý định (TF-IDF + LogisticRegression) cho trợ lý ảo tiếng Việt.

**Có gì mới ở v2:**
- Thêm nhãn `unknown` cho câu chào hỏi/tán gẫu ngoài phạm vi (tránh mô hình ép nhãn sai).
- Tự động sinh thêm câu KHÔNG DẤU để nhận diện tốt hơn khi STT/gõ tắt không dấu.
- Dùng `rapidfuzz` để so khớp tên app/web/file khoan dung lỗi chính tả nhẹ (ở executor.py, không ảnh hưởng notebook này).

**Lưu ý:** Colab chỉ dùng để HUẤN LUYỆN. Phần thực thi lệnh (mở app, tắt máy) phải chạy trên máy Windows của bạn.

Chạy lần lượt từng cell bằng `Shift + Enter`.

## Bước 1. Cài thư viện (Colab thường đã có sẵn)

In [ ]:
!pip install -q scikit-learn pandas joblib rapidfuzz
import sklearn, pandas
print('scikit-learn:', sklearn.__version__)
print('pandas:', pandas.__version__)

## Bước 2. Tải code lên Colab

Chọn MỘT trong hai cách bên dưới.

### Cách A: Upload file zip `vi_voice_assistant.zip`

In [ ]:
from google.colab import files
up = files.upload()   # chọn file vi_voice_assistant.zip
!unzip -o vi_voice_assistant.zip
%cd vi_voice_assistant

### Cách B: Không upload gì cả — dán dataset trực tiếp vào notebook

> ⚠️ **Lưu ý (v2):** Đây là bản SAO RÚT GỌN của `dataset.py`, dùng khi bạn không muốn upload file nào.
> Bản này **không có** nhãn `unknown` và không tự sinh câu không dấu như `dataset.py` chính thức,
> nên độ chính xác/độ an toàn sẽ THẤP HƠN so với Cách A. Khuyến nghị dùng **Cách A** (upload zip)
> để dùng đúng `dataset.py` + `text_utils.py` mới nhất. Chỉ dùng Cách B để thử nghiệm nhanh.

Chạy cell dưới nếu bạn muốn tự tạo lại dataset ngay trong Colab.

In [ ]:
INTENT_DATA = {
    "open_website": [
        "bật google lên", "mở google giúp tôi", "vào trang youtube đi",
        "mở youtube lên xem phim", "cho tôi xem facebook", "truy cập facebook nào",
        "mở web github", "vào github giúp tôi", "lên mạng vào trang gmail",
        "mở gmail kiểm tra thư", "mở trang chủ notion", "truy cập vào website chatgpt",
        "bật trình duyệt vào google dịch", "mở tiki lên mua đồ", "vào shopee xem hàng",
        "mở trang tin vnexpress", "cho tôi vào trang zalo web", "lên youtube nghe nhạc",
        "mở web stackoverflow", "vào trang chủ của google",
    ],
    "open_app": [
        "mở chrome ra", "khởi chạy chrome giúp tôi", "bật notepad lên",
        "mở ứng dụng notepad", "mở máy tính bỏ túi", "bật calculator lên tính toán",
        "mở vs code lên code", "khởi động visual studio code", "mở phần mềm word",
        "bật excel lên nhập số liệu", "mở powerpoint làm slide", "chạy ứng dụng cmd",
        "mở cửa sổ dòng lệnh cmd", "bật paint lên vẽ", "mở phần mềm zalo",
        "khởi chạy ứng dụng spotify", "mở file explorer", "bật task manager lên",
        "mở ứng dụng telegram", "chạy chương trình notepad giúp tôi",
    ],
    "open_file": [
        "mở file báo cáo", "cho tôi xem file báo cáo tháng này", "mở tài liệu word ra",
        "mở file tài liệu giúp tôi", "mở ảnh chụp màn hình", "cho tôi xem file ảnh",
        "mở file excel doanh thu", "mở file pdf hợp đồng", "mở thư mục tải xuống",
        "mở file ghi chú của tôi", "mở file danh sách sinh viên", "xem lại file luận văn",
        "mở tệp tin bài tập", "mở file nhạc trong máy", "mở file video quay màn hình",
        "mở tài liệu hướng dẫn", "mở file dữ liệu csv", "cho tôi mở tệp báo cáo tài chính",
    ],
    "system_control": [
        "tắt máy tính đi", "shutdown máy giúp tôi", "tắt nguồn máy tính",
        "khởi động lại máy", "restart máy tính giúp tôi", "reboot lại hệ thống",
        "khoá màn hình lại", "lock máy tính", "cho máy ngủ đi",
        "chuyển sang chế độ sleep", "tắt âm thanh", "tắt tiếng loa đi",
        "bật âm thanh lên", "mở tiếng lên giúp tôi", "tăng âm lượng",
        "giảm âm lượng xuống", "đăng xuất tài khoản", "thoát khỏi máy tính",
        "tắt máy sau 5 phút", "khởi động lại windows",
    ],
}

import pandas as pd
rows = [(s, i) for i, ss in INTENT_DATA.items() for s in ss]
df = pd.DataFrame(rows, columns=["text", "intent"])
print("Tổng số câu:", len(df))
df.groupby("intent").size()

## Bước 3. Huấn luyện mô hình

Nếu bạn dùng **Cách A** (đã upload zip), chỉ cần chạy 1 dòng dưới đây.

In [ ]:
!python intent_model.py

Nếu bạn dùng **Cách B** (không upload code), chạy cell dưới để huấn luyện trực tiếp:

In [ ]:
import re, json, unicodedata, joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def normalize_text(t):
    t = unicodedata.normalize("NFC", t.strip().lower())
    t = re.sub(r"[^\w\s\.\-/:\\]", " ", t, flags=re.UNICODE)
    return re.sub(r"\s+", " ", t).strip()

X = [normalize_text(t) for t in df["text"]]
y = list(df["intent"])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, C=10)),
])
pipe.fit(Xtr, ytr)
print(classification_report(yte, pipe.predict(Xte), zero_division=0))

# Huấn luyện lại trên toàn bộ dữ liệu rồi lưu model
pipe.fit(X, y)
joblib.dump(pipe, "intent_model.pkl")
print("Đã lưu intent_model.pkl")

## Bước 4. Thử dự đoán (không thực thi lệnh)

In [ ]:
import joblib, json
model = joblib.load("intent_model.pkl")

def predict(text):
    c = normalize_text(text)
    intent = model.predict([c])[0]
    conf = float(max(model.predict_proba([c])[0]))
    return {"intent": intent, "target": c, "confidence": round(conf, 4)}

for s in ["bật google lên", "mở vs code", "mở file báo cáo", "tắt tiếng loa"]:
    print(s, "->", json.dumps(predict(s), ensure_ascii=False))

## Bước 5. Tải model đã huấn luyện về máy

Sau đó chép `intent_model.pkl` vào cùng thư mục với `main.py` trên máy Windows và chạy `python main.py`.

In [ ]:
from google.colab import files
files.download('intent_model.pkl')

### (Tuỳ chọn) Lưu vào Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp intent_model.pkl /content/drive/MyDrive/

## Lưu ý về phiên bản

File `.pkl` nên được load bằng **cùng phiên bản scikit-learn**. Kiểm tra bằng `sklearn.__version__` ở Bước 1 và cài đúng bản đó trên máy:

```
pip install scikit-learn==<phiên bản>
```

Nếu ngại, bạn có thể huấn luyện thẳng trên máy Windows: `python intent_model.py` (chỉ mất vài giây).
